<a href="https://colab.research.google.com/github/vikiiiiiiiiiiii/Global-India-AI-Model-Intelligence-Registry/blob/main/Global_India_AI_Model_Intelligence_Registry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langgraph langchain-google-genai pydantic requests pandas gradio

import os
os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"
print("✅ Environment configured!")

In [ ]:
import sqlite3
import json
from typing import List, Optional, TypedDict
from pydantic import BaseModel, Field

# 1. Strict Data Schema for each AI Model
class AIModelRecord(BaseModel):
    model_id: str
    name: str
    provider: str
    context_window: int
    modalities: List[str]
    # Pricing in INR per 1 Million Tokens
    input_price_inr: float
    output_price_inr: float
    input_price_inr_with_gst: float
    output_price_inr_with_gst: float
    # Benchmarks & India Sovereignty Metadata
    coding_score: float = Field(description="Normalized score out of 100")
    reasoning_score: float = Field(description="Normalized score out of 100")
    indic_support: str = Field(description="High, Moderate, or Basic")
    india_regions: List[str] = Field(description="e.g., ['AWS Mumbai (ap-south-1)', 'Azure Central India (Pune)']")
    data_residency_compliant: bool
    deployment_options: List[str] = Field(description="e.g., ['Serverless API', 'VPC Dedicated', 'Open-Weights Self-Host']")
    best_enterprise_use_cases: List[str]

class EnrichedRegistryBatch(BaseModel):
    models: List[AIModelRecord]

# 2. Initialize Local SQLite Registry
DB_PATH = "global_ai_model_registry.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS models (
            model_id TEXT PRIMARY KEY,
            name TEXT,
            provider TEXT,
            context_window INTEGER,
            modalities TEXT,
            input_price_inr REAL,
            output_price_inr REAL,
            input_price_inr_with_gst REAL,
            output_price_inr_with_gst REAL,
            coding_score REAL,
            reasoning_score REAL,
            indic_support TEXT,
            india_regions TEXT,
            data_residency_compliant INTEGER,
            deployment_options TEXT,
            best_enterprise_use_cases TEXT,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

init_db()
print("✅ SQLite Registry Database initialized.")

In [ ]:
import requests

def fetch_live_usd_to_inr() -> float:
    """Fetches real-time USD to INR exchange rate."""
    try:
        res = requests.get("https://open.er-api.com/v6/latest/USD", timeout=10).json()
        return round(float(res["rates"]["INR"]), 2)
    except Exception:
        return 84.50  # Fallback rate

def fetch_newly_released_models(limit: int = 12) -> list:
    """Autonomously discovers newly released models from OpenRouter & Hugging Face."""
    discovered = []
    UNSAFE_KEYWORDS = ("uncensored", "abliterated", "heretic", "jailbreak")

    # Source 1: OpenRouter Live API (sorted by newest)
    try:
        resp = requests.get("https://openrouter.ai/api/v1/models", timeout=15).json()
        raw_models = resp.get("data", [])
        newest = sorted(raw_models, key=lambda x: x.get("created", 0), reverse=True)[:limit]

        for m in newest:
            mid = m.get("id", "")
            if any(k in mid.lower() for k in UNSAFE_KEYWORDS):
                continue
            pricing = m.get("pricing", {})
            prompt_usd = float(pricing.get("prompt", 0)) * 1_000_000
            completion_usd = float(pricing.get("completion", 0)) * 1_000_000

            discovered.append({
                "model_id": mid,
                "name": m.get("name"),
                "provider": mid.split("/")[0].capitalize(),
                "context_window": m.get("context_length", 32000),
                "description": m.get("description", "")[:300],
                "prompt_usd_per_1m": round(prompt_usd, 4),
                "completion_usd_per_1m": round(completion_usd, 4),
                "architecture": m.get("architecture", {}).get("modality", "text->text")
            })
    except Exception as e:
        print(f"OpenRouter fetch warning: {e}")

    # Source 2: Hugging Face Hub (Enterprise-safe open-weight text-generation models)
    try:
        hf_resp = requests.get(
            "https://huggingface.co/api/models?pipeline_tag=text-generation&sort=createdAt&direction=-1&limit=8",
            timeout=10
        ).json()
        for hf in hf_resp:
            mid = hf.get("id", "")
            if any(k in mid.lower() for k in UNSAFE_KEYWORDS):
                continue
            discovered.append({
                "model_id": mid,
                "name": mid.split("/")[-1],
                "provider": mid.split("/")[0],
                "context_window": 128000,
                "description": f"Open-weights release on Hugging Face. Tags: {', '.join(hf.get('tags', [])[:5])}",
                "prompt_usd_per_1m": 0.0,
                "completion_usd_per_1m": 0.0,
                "architecture": "open-weights"
            })
    except Exception as e:
        print(f"Hugging Face fetch warning: {e}")

    return discovered

# Quick test
fx = fetch_live_usd_to_inr()
sample = fetch_newly_released_models(limit=5)
print(f"✅ Live FX Rate: 1 USD = ₹{fx} | Discovered {len(sample)} enterprise-safe models.")

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI

# Define Shared Agent State
class RegistryPipelineState(TypedDict):
    fx_rate: float
    raw_models: list
    enriched_models: list
    logs: list

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
structured_auditor_llm = llm.with_structured_output(EnrichedRegistryBatch)

# --- AGENT 1: Autonomous Discovery Scout ---
def discovery_scout_node(state: RegistryPipelineState):
    fx = fetch_live_usd_to_inr()
    raw = fetch_newly_released_models(limit=10)
    log_msg = f"[Discovery Scout] Fetched {len(raw)} newly released models worldwide. Live FX: ₹{fx}/USD."
    return {"fx_rate": fx, "raw_models": raw, "logs": state.get("logs", []) + [log_msg]}

# --- AGENT 2: INR Pricing & Indian GST Normalizer ---
def inr_pricing_node(state: RegistryPipelineState):
    fx = state["fx_rate"]
    GST_MULTIPLIER = 1.18  # 18% Indian GST on SaaS/API services
    priced_models = []

    for m in state["raw_models"]:
        in_inr = round(m["prompt_usd_per_1m"] * fx, 2)
        out_inr = round(m["completion_usd_per_1m"] * fx, 2)
        m["input_price_inr"] = in_inr
        m["output_price_inr"] = out_inr
        m["input_price_inr_with_gst"] = round(in_inr * GST_MULTIPLIER, 2)
        m["output_price_inr_with_gst"] = round(out_inr * GST_MULTIPLIER, 2)
        priced_models.append(m)

    log_msg = f"[INR Pricing Agent] Converted all token rates to INR and applied 18% GST slabs."
    return {"raw_models": priced_models, "logs": state["logs"] + [log_msg]}

# --- AGENT 3: India Sovereignty, Cloud Regions & Benchmark Auditor ---
def india_auditor_node(state: RegistryPipelineState):
    prompt = f"""
    You are an Enterprise AI Solutions Architect specializing in India deployments (RBI/DPDP Act compliance).
    Analyze these newly discovered AI models with pre-calculated INR pricing and enrich each record:

    1. Map realistic India cloud availability (e.g., AWS ap-south-1 Mumbai, Azure Central India Pune, GCP asia-south1 Mumbai, Vertex AI India, or Global API Only).
    2. Set `data_residency_compliant` to True ONLY if hosted in Indian data centers or open-weights self-hostable on Indian VPC/on-prem.
    3. Estimate realistic `coding_score` (0-100) and `reasoning_score` (0-100) based on model family/size.
    4. Assess `indic_support` (High, Moderate, or Basic) for Hindi/Tamil/Telugu/Indic languages.
    5. Assign 2-3 `best_enterprise_use_cases` from:
       ['BFSI & Regulated Compliance', 'Agentic Coding & DevOps', 'Indic Multilingual Customer Support', 'High-Volume Document Extraction', 'Enterprise RAG & Legal Synthesis'].

    Input Models JSON:
    {json.dumps(state['raw_models'], indent=2)}
    """
    result: EnrichedRegistryBatch = structured_auditor_llm.invoke(prompt)
    enriched_dicts = [m.model_dump() for m in result.models]

    log_msg = f"[India Sovereignty & Benchmark Auditor] Audited {len(enriched_dicts)} models for DPDP residency & use cases."
    return {"enriched_models": enriched_dicts, "logs": state["logs"] + [log_msg]}

# --- AGENT 4: SQLite Registry Persistence ---
def persist_registry_node(state: RegistryPipelineState):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    for m in state["enriched_models"]:
        cursor.execute("""
            INSERT OR REPLACE INTO models (
                model_id, name, provider, context_window, modalities,
                input_price_inr, output_price_inr, input_price_inr_with_gst, output_price_inr_with_gst,
                coding_score, reasoning_score, indic_support, india_regions,
                data_residency_compliant, deployment_options, best_enterprise_use_cases
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            m["model_id"], m["name"], m["provider"], m["context_window"], json.dumps(m["modalities"]),
            m["input_price_inr"], m["output_price_inr"], m["input_price_inr_with_gst"], m["output_price_inr_with_gst"],
            m["coding_score"], m["reasoning_score"], m["indic_support"], json.dumps(m["india_regions"]),
            1 if m["data_residency_compliant"] else 0, json.dumps(m["deployment_options"]),
            json.dumps(m["best_enterprise_use_cases"])
        ))
    conn.commit()
    conn.close()

    log_msg = f"[Registry DB] Persisted {len(state['enriched_models'])} models into SQLite registry."
    return {"logs": state["logs"] + [log_msg]}

# Compile LangGraph Workflow
workflow = StateGraph(RegistryPipelineState)
workflow.add_node("scout", discovery_scout_node)
workflow.add_node("pricer", inr_pricing_node)
workflow.add_node("auditor", india_auditor_node)
workflow.add_node("persister", persist_registry_node)

workflow.set_entry_point("scout")
workflow.add_edge("scout", "pricer")
workflow.add_edge("pricer", "auditor")
workflow.add_edge("auditor", "persister")
workflow.add_edge("persister", END)

autonomous_pipeline = workflow.compile()
print("✅ Multi-Agent LangGraph Pipeline compiled!")

In [ ]:
import pandas as pd
from datetime import date
from langchain_google_genai import ChatGoogleGenerativeAI

# Fast LLM client with strict timeout so Gradio never hangs
fast_report_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    timeout=25,
    max_retries=1
)

def build_model_source_url(model_id: str, deployment_str: str) -> str:
    """Constructs a direct verification URL for the discovered model."""
    if "Open-Weights" in deployment_str:
        return f"https://huggingface.co/{model_id}"
    return f"https://openrouter.ai/{model_id}"

def get_use_case_sources(use_case: str, primary: dict, fallback: dict) -> str:
    """Returns workload-specific authoritative web sources backing the recommendation."""
    base_sources = [
        f"- **Primary Recommended Model Spec & Pricing:** [{primary['Model Name']}]({primary['Source URL']})",
        f"- **Fallback / Alternative Model Spec & Pricing:** [{fallback['Model Name']}]({fallback['Source URL']})",
        "- **Commercial Model Catalog & Token Pricing:** [OpenRouter Models Directory](https://openrouter.ai/models)",
        "- **Open-Weight Model Cards & Licensing:** [Hugging Face Model Hub](https://huggingface.co/models)",
        "- **Live USD-to-INR Exchange Rate Reference:** [Open Exchange Rates Portal](https://www.exchangerate-api.com)",
        "- **Indian Enterprise Tax Compliance (18% GST):** [Goods and Services Tax (GST) Portal](https://www.gst.gov.in)"
    ]

    # Add cloud data residency & regulatory compliance sources
    residency_sources = [
        "- **GCP Mumbai (`asia-south1`) Data Residency Docs:** [Google Cloud Vertex AI Data Residency](https://cloud.google.com/vertex-ai/docs/general/data-residency)",
        "- **AWS Mumbai (`ap-south-1`) Data Residency Docs:** [AWS Bedrock Data Protection](https://docs.aws.amazon.com/bedrock/latest/userguide/data-protection.html)",
        "- **Azure Central India (Pune) Residency Docs:** [Azure OpenAI Data Privacy & Residency](https://learn.microsoft.com/en-us/legal/cognitive-services/openai/data-privacy)"
    ]

    # Add domain-specific regulatory/benchmark sources based on selected workload
    if "BFSI" in use_case or "Compliance" in use_case:
        domain_sources = [
            "- **India Data Privacy & Localization Law:** [MeitY Digital Personal Data Protection (DPDP) Act Framework](https://www.meity.gov.in/data-protection-framework)",
            "- **Financial Sector Data Localization Mandate:** [Reserve Bank of India (RBI) Notifications](https://www.rbi.org.in)"
        ]
    elif "Coding" in use_case or "Software" in use_case:
        domain_sources = [
            "- **Software Engineering Benchmark Reference:** [SWE-bench Leaderboards](https://www.swebench.com)",
            "- **India Data Privacy Framework:** [MeitY DPDP Act Framework](https://www.meity.gov.in/data-protection-framework)"
        ]
    else:
        domain_sources = [
            "- **LLM Reasoning & Capability Benchmarks:** [LMSYS Chatbot Arena Leaderboard](https://lmarena.ai)",
            "- **India Data Privacy Framework:** [MeitY DPDP Act Framework](https://www.meity.gov.in/data-protection-framework)"
        ]

    return "\n".join(base_sources + residency_sources + domain_sources)

def get_registry_df(india_only: bool = False, include_gst: bool = True) -> pd.DataFrame:
    conn = sqlite3.connect(DB_PATH)
    query = "SELECT * FROM models"
    if india_only:
        query += " WHERE data_residency_compliant = 1"
    df = pd.read_sql_query(query, conn)
    conn.close()

    if df.empty:
        return df

    in_col = "input_price_inr_with_gst" if include_gst else "input_price_inr"
    out_col = "output_price_inr_with_gst" if include_gst else "output_price_inr"

    deployment_series = df["deployment_options"].apply(lambda x: ", ".join(json.loads(x)))
    source_urls = [
        build_model_source_url(mid, dep)
        for mid, dep in zip(df["model_id"], deployment_series)
    ]

    display_df = pd.DataFrame({
        "Model Name": df["name"],
        "Provider": df["provider"],
        "Context": df["context_window"],
        "Input ₹ / 1M": df[in_col],
        "Output ₹ / 1M": df[out_col],
        "Coding": df["coding_score"],
        "Reasoning": df["reasoning_score"],
        "Indic Support": df["indic_support"],
        "India Data Residency": df["data_residency_compliant"].apply(lambda x: "✅ Compliant" if x == 1 else "❌ Global Only"),
        "India Regions": df["india_regions"].apply(lambda x: ", ".join(json.loads(x))),
        "Deployment": deployment_series,
        "Best Use Cases": df["best_enterprise_use_cases"].apply(lambda x: ", ".join(json.loads(x))),
        "Source URL": source_urls
    })
    return display_df

def build_instant_fallback_report(use_case: str, vol: float, residency: bool, top_models: list, today_str: str) -> str:
    primary = top_models[0]
    fallback = top_models[1] if len(top_models) > 1 else primary
    sources_md = get_use_case_sources(use_case, primary, fallback)

    rows = "\n".join([
        f"| [{m['Model Name']}]({m['Source URL']}) | {m['Provider']} | {m['Reasoning']}/100 | {m['Coding']}/100 | ₹{m['Est. Monthly Spend (₹ incl. GST)']:,.2f} | {m['India Regions']} |"
        for m in top_models
    ])

    return f"""# Enterprise AI Model Comparative & Recommendation Report
**Date:** {today_str} | **Workload:** {use_case} | **Volume:** {vol}M Tokens/mo | **India Residency Enforced:** {'Yes (Mumbai/Pune)' if residency else 'No'}

## 1. Executive Recommendation
* **Primary Recommended Model:** **[{primary['Model Name']}]({primary['Source URL']})** ({primary['Provider']}) — Reasoning: **{primary['Reasoning']}/100**, Coding: **{primary['Coding']}/100**, Projected Spend: **₹{primary['Est. Monthly Spend (₹ incl. GST)']:,.2f}/month** (incl. 18% GST).
* **Cost-Optimized Fallback:** **[{fallback['Model Name']}]({fallback['Source URL']})** ({fallback['Provider']}) — Projected Spend: **₹{fallback['Est. Monthly Spend (₹ incl. GST)']:,.2f}/month** (incl. 18% GST).

## 2. Comparative TCO & Benchmark Matrix (INR incl. 18% GST)
| Model | Provider | Reasoning | Coding | Est. Monthly Spend (₹) | India Cloud Availability |
| :--- | :--- | :---: | :---: | :---: | :--- |
{rows}

## 3. Deployment & DPDP Act Compliance Architecture
* **Hosting Topology:** Deploy **[{primary['Model Name']}]({primary['Source URL']})** via **{primary['Deployment']}** within **{primary['India Regions']}** to ensure zero cross-border PII egress under India's Digital Personal Data Protection (DPDP) Act.

## 4. Core Information & Compliance Reference Sources
{sources_md}
"""

def generate_enterprise_report(use_case: str, monthly_tokens_millions: float, require_india_residency: bool) -> tuple[str, str]:
    today_str = date.today().strftime("%B %d, %Y")
    df = get_registry_df(india_only=require_india_residency, include_gst=True)
    if df.empty:
        return "⚠️ No models found matching criteria. Run an Autonomous Discovery Scan first.", None

    df["Est. Monthly Spend (₹ incl. GST)"] = round(
        (df["Input ₹ / 1M"] * 0.7 + df["Output ₹ / 1M"] * 0.3) * monthly_tokens_millions, 2
    )

    if "Coding" in use_case:
        df = df.sort_values(by=["Coding", "Est. Monthly Spend (₹ incl. GST)"], ascending=[False, True])
    elif "Extraction" in use_case:
        df = df.sort_values(by=["Est. Monthly Spend (₹ incl. GST)", "Reasoning"], ascending=[True, False])
    else:
        df = df.sort_values(by=["Reasoning", "Est. Monthly Spend (₹ incl. GST)"], ascending=[False, True])

    top_models = df.head(5).to_dict(orient="records")
    primary = top_models[0]
    fallback = top_models[1] if len(top_models) > 1 else primary
    sources_md = get_use_case_sources(use_case, primary, fallback)

    report_prompt = f"""
    Generate a crisp, executive Enterprise AI Model Comparative & Recommendation Report.
    - Report Date: {today_str}
    - Target Enterprise Use Case: {use_case}
    - Projected Monthly Volume: {monthly_tokens_millions} Million Tokens (70% Input / 30% Output)
    - Strict India Data Residency Required: {require_india_residency}

    Candidate Models from Live Registry (All costs in INR ₹ including 18% GST):
    {json.dumps(top_models, indent=2)}

    Structure the report with these exact 4 sections:
    1. Executive Recommendation (Top Primary Model & Cost-Optimized Fallback, with clickable Markdown links using each model's Source URL and citing the relevant regulatory/pricing basis)
    2. Comparative TCO & Benchmark Table (Markdown table with clickable model links, INR monthly cost, context, and India regions)
    3. Deployment & DPDP Act Compliance Architecture (VPC vs Serverless API in Mumbai/Pune)
    4. Core Information & Compliance Reference Sources (Include the exact Markdown bullet list below verbatim so readers can verify every pricing, benchmark, cloud residency, and statutory source):
    {sources_md}
    """

    try:
        report_md = fast_report_llm.invoke(report_prompt).content
    except Exception:
        report_md = build_instant_fallback_report(use_case, monthly_tokens_millions, require_india_residency, top_models, today_str)

    report_path = "/content/Enterprise_AI_Model_Report.md"
    with open(report_path, "w") as f:
        f.write(report_md)

    return report_md, report_path

print("✅ Updated Cell 5 ready (includes workload-specific core information & compliance reference links).")

In [ ]:
import gradio as gr

def run_agent_scan_ui(india_only, include_gst):
    result = autonomous_pipeline.invoke({"logs": []})
    logs_text = "\n".join(result["logs"])
    df = get_registry_df(india_only=india_only, include_gst=include_gst)
    return logs_text, df

def refresh_table_ui(india_only, include_gst):
    return get_registry_df(india_only=india_only, include_gst=include_gst)

with gr.Blocks(theme=gr.themes.Soft(), title="Global & India AI Model Intelligence Registry") as dashboard:
    gr.Markdown("# 🌐 Global & India AI Model Intelligence Registry (Agentic Control Center)")
    gr.Markdown("Autonomous multi-agent discovery, live INR + 18% GST cost normalization, India data residency tracking, and enterprise TCO reporting.")

    with gr.Tabs():
        # TAB 1: Live Registry & Autonomous Discovery
        with gr.TabItem("📡 Live Model Registry & Agent Discovery"):
            with gr.Row():
                scan_btn = gr.Button("🚀 Trigger Autonomous Discovery Scan", variant="primary")
                india_filter = gr.Checkbox(label="🇮🇳 India Data Residency Compliant Only", value=False)
                gst_toggle = gr.Checkbox(label="Include 18% Indian Enterprise GST", value=True)

            agent_logs = gr.Textbox(label="Autonomous Agent Pipeline Telemetry", lines=4, interactive=False)
            registry_table = gr.Dataframe(label="Discovered AI Models (Live SQLite Registry)", wrap=True)

            scan_btn.click(
                fn=run_agent_scan_ui,
                inputs=[india_filter, gst_toggle],
                outputs=[agent_logs, registry_table]
            )
            india_filter.change(fn=refresh_table_ui, inputs=[india_filter, gst_toggle], outputs=registry_table)
            gst_toggle.change(fn=refresh_table_ui, inputs=[india_filter, gst_toggle], outputs=registry_table)

        # TAB 2: Enterprise Use-Case & Comparative Report Generator
        with gr.TabItem("📊 Enterprise Use-Case Recommendation & Report"):
            with gr.Row():
                use_case_dd = gr.Dropdown(
                    choices=[
                        "BFSI & Regulated Compliance",
                        "Agentic Coding & DevOps",
                        "Indic Multilingual Customer Support",
                        "High-Volume Document Extraction",
                        "Enterprise RAG & Legal Synthesis"
                    ],
                    value="BFSI & Regulated Compliance",
                    label="Select Enterprise Workload"
                )
                volume_slider = gr.Slider(minimum=10, maximum=5000, value=250, step=10, label="Monthly Token Volume (Millions)")
                residency_req = gr.Checkbox(label="Enforce India Data Residency (Mumbai/Pune)", value=True)

            gen_report_btn = gr.Button("📑 Generate Comparative Enterprise Report", variant="primary")
            download_file = gr.File(label="Download Report (.md)")
            report_output = gr.Markdown(label="Comparative Recommendation Report")

            gen_report_btn.click(
                fn=generate_enterprise_report,
                inputs=[use_case_dd, volume_slider, residency_req],
                outputs=[report_output, download_file]
            )

# Launch inside Colab
dashboard.launch(debug=True, share=True)